# MolmoAct2 - LIBERO interactive simulation (synchronous)

Run a fine-tuned **MolmoAct2 LIBERO policy** in the simulator and render it **inline, right here in the notebook**. By default, the notebook uses the newest checkpoint produced by notebook 1. In the setup cell, change `USE_PUBLIC_CHECKPOINT = False` to `True` to download AllenAI's fully trained public `allenai/MolmoAct2-LIBERO` checkpoint for comparison.

This is the **synchronous** rollout: the policy plans an action chunk, the simulator executes it, then the policy re-plans - the same deterministic closed-loop (receding-horizon) control used by the Step-5 LIBERO eval. The arm pauses briefly to "think" between chunks, but the motion is precise and there is **no real-time blending** (blending can occasionally jostle a grasped object loose).

There is **no extra port to forward**: the sim's small web UI is served inside your session and proxied through your JupyterHub route via `jupyter-server-proxy`, then embedded below with an `IFrame`. Type an instruction in the panel, press **Send**, and watch the policy act; the status line shows the executed command, step count, and live inference latency.

**Checkpoint choice:** set `USE_PUBLIC_CHECKPOINT = True` in the setup cell to download the public comparison checkpoint. Other overrides: `POLICY_PATH=/path/to/a/LeRobot-checkpoint`, `RELEASED_CKPT=<Hugging-Face-model-id>`, `SUITE=libero_object`, `TASK_ID=3`, `RT_PORT=8080`.

## Goals

* Load your locally fine-tuned policy or the public MolmoAct2 LIBERO release and serve it behind a small web UI inside your JupyterHub session
* Command the policy in **natural language** and watch it act in the LIBERO simulator, rendered inline in the notebook
* Understand the **synchronous receding-horizon** control loop: plan an action chunk, execute it, re-plan - and why this deterministic path avoids the real-time blending that can jostle a grasped object loose
* See how `jupyter-server-proxy` embeds a GPU-rendered simulator with no extra port to forward

In [ ]:
import glob
import os
import subprocess
import json
import sys
import time
import urllib.request

_SCRIPTS = "/ryzers/notebooks/scripts"
if _SCRIPTS not in sys.path:
    sys.path.insert(0, _SCRIPTS)
from ft_helpers import prefetch

# Quiet ROCm/torch spam in any child process we spawn.
os.environ.setdefault("TORCH_BLAS_PREFER_HIPBLASLT", "0")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

OUT_DIR = os.environ.get("OUT_DIR", "/outputs")
os.makedirs(OUT_DIR, exist_ok=True)


# Subprocesses (the sim server) inherit a clean env: drop the notebook's inline matplotlib
# backend (crashes headless children) and force plain hipBLAS + unbuffered output.
def child_env(**extra):
    e = dict(os.environ)
    e.pop("MPLBACKEND", None)
    e.update({"TORCH_BLAS_PREFER_HIPBLASLT": "0", "PYTHONWARNINGS": "ignore", "PYTHONUNBUFFERED": "1"})
    e.update({k: str(v) for k, v in extra.items()})
    return e


# Optional compatibility path for an organizer-provided offline cache. Home users leave
# ASSETS_DIR unset; the released fallback downloads directly from Hugging Face.
ASSETS_DIR = os.environ.get("ASSETS_DIR", "").strip()


def _stage_assets():
    import shutil
    if not ASSETS_DIR or not os.path.isdir(ASSETS_DIR):
        return
    hub = os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "hub")
    hub_src = os.path.join(ASSETS_DIR, "hf_hub")
    if os.path.isdir(hub_src):
        os.makedirs(hub, exist_ok=True)
        for _n in sorted(os.listdir(hub_src)):
            _s, _d = os.path.join(hub_src, _n), os.path.join(hub, _n)
            if os.path.isdir(_s) and not os.path.exists(_d):
                print(f"staging {_n} -> HF cache")
                shutil.copytree(_s, _d, symlinks=True)
    _rs = os.path.join(ASSETS_DIR, "checkpoints", "reference", "pretrained_model")
    _rd = os.environ.get("REFERENCE_POLICY", os.path.expanduser("~/checkpoints/reference/pretrained_model"))
    if os.path.isdir(_rs) and not os.path.isdir(_rd):
        print(f"staging fine-tuned checkpoint -> {_rd}")
        os.makedirs(os.path.dirname(_rd), exist_ok=True)
        shutil.copytree(_rs, _rd, symlinks=True)


if ASSETS_DIR:
    print(f"staging assets from {ASSETS_DIR} ...")
    _stage_assets()


def _is_ckpt(p):
    return bool(p) and os.path.isdir(p) and os.path.exists(os.path.join(p, "config.json"))


# Prefer notebook 1's newest training output and make no model download by default. Change this
# visible notebook setting to True to download AllenAI's fully trained public comparison policy;
# an explicit POLICY_PATH always wins.
CKPT_ROOT = os.environ.get("CHECKPOINTS_DIR", os.path.expanduser("~/checkpoints"))
REFERENCE = os.environ.get("REFERENCE_POLICY", os.path.join(CKPT_ROOT, "reference", "pretrained_model"))
RELEASED_REPO = os.environ.get("RELEASED_CKPT", "allenai/MolmoAct2-LIBERO")
RELEASED_REVISION = os.environ.get("RELEASED_REVISION", "0d24a92bd1faf321ef497c3bbd5681af97c65aa2")
USE_PUBLIC_CHECKPOINT = False  # Change to True to download the public comparison checkpoint.
RELEASED_CKPT = RELEASED_REPO
_explicit = os.environ.get("POLICY_PATH", "").strip()
_trained = sorted(glob.glob(os.path.join(CKPT_ROOT, "*", "checkpoints", "*", "pretrained_model")))
_trained = [p for p in _trained if os.path.basename(os.path.dirname(p)) != "last"] or _trained

if _explicit:
    POLICY_PATH = _explicit
    POLICY_SOURCE = POLICY_PATH
    print("interactive sim will load POLICY_PATH (from env):", POLICY_SOURCE)
elif USE_PUBLIC_CHECKPOINT:
    POLICY_PATH = ""
    print("comparison mode: downloading pinned fully trained public release:", RELEASED_REPO)
    RELEASED_CKPT = prefetch(RELEASED_REPO, "model", revision=RELEASED_REVISION)
    POLICY_SOURCE = f"{RELEASED_REPO}@{RELEASED_REVISION[:12]}"
elif _trained:
    POLICY_PATH = _trained[-1]
    POLICY_SOURCE = POLICY_PATH
    print("interactive sim will load notebook 1's newest checkpoint:", POLICY_SOURCE)
elif _is_ckpt(REFERENCE):
    POLICY_PATH = REFERENCE
    POLICY_SOURCE = POLICY_PATH
    print("interactive sim will load offline REFERENCE_POLICY:", POLICY_SOURCE)
else:
    raise FileNotFoundError(
        "No local checkpoint from notebook 1 was found. Run notebook 1 first, set POLICY_PATH to "
        "a compatible local checkpoint, or change USE_PUBLIC_CHECKPOINT = True in this cell to "
        "download AllenAI's fully trained public comparison checkpoint."
    )

## Launch the interactive simulator

The cell below starts the sim server inside your pod, waits for the policy to load, then embeds the live simulator right here in the notebook. The first run JIT-compiles ROCm kernels, so give it a couple of minutes before the panel appears.

Once it's up, pick a task from the **environment** dropdown, type an instruction, and press **Send**. Each instruction is planned and executed by the selected policy - the status line shows the executed command, the step count, and the live inference latency per plan.

In [ ]:
from IPython.display import IFrame, display

# Internal port inside your pod; reached only through the authenticated JupyterHub proxy
# (never exposed directly). The server binds 0.0.0.0:RT_PORT; jupyter-server-proxy forwards
# {JUPYTERHUB_SERVICE_PREFIX}/proxy/RT_PORT/ -> 127.0.0.1:RT_PORT.
RT_PORT = os.environ.get("RT_PORT", "8080")
SIM_SERVER = "/ryzers/notebooks/scripts/interactive_server_ft.py"

# Stop a server we started from a previous run of this cell.
try:
    if globals().get("_sim") and _sim.poll() is None:
        _sim.terminate()
        _sim.wait(timeout=5)
except Exception:
    pass

# Serve a local LeRobot checkpoint through POLICY_PATH, or use the server's released-checkpoint
# fallback through CKPT. Stream logs to a file so the kernel cannot block on a full stdout pipe.
sim_env = child_env(
    POLICY_PATH=POLICY_PATH,
    CKPT=RELEASED_CKPT,
    PORT=RT_PORT,
    SUITE=os.environ.get("SUITE", "libero_object"),
    TASK_ID=os.environ.get("TASK_ID", "3"),
)
_log_path = os.path.join(OUT_DIR, "interactive_rt.log")
_log = open(_log_path, "w")
_sim = subprocess.Popen(
    ["/opt/train-venv/bin/python", SIM_SERVER],
    env=sim_env, stdout=_log, stderr=subprocess.STDOUT,
)
print(f"sim server started (pid {_sim.pid}); loading policy: {POLICY_SOURCE}")
print("(the first public download and first kernel JIT can each take several minutes)")


def _sim_status():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{RT_PORT}/status", timeout=3) as r:
            return json.load(r)
    except Exception:
        return None


ready, deadline = False, time.time() + 900
while time.time() < deadline:
    if _sim.poll() is not None:
        print("\nserver exited early; tail of log:")
        print("".join(open(_log_path).readlines()[-20:]))
        break
    s = _sim_status()
    if s:
        print("  " + str(s.get("status", ""))[:90], end="\r")
        if s.get("mode") in ("idle", "running"):
            ready = True
            break
        if s.get("mode") == "error":
            print("\nserver error:", s.get("status"))
            break
    time.sleep(3)

prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "").rstrip("/")
sim_url = f"{prefix}/proxy/{RT_PORT}/" if prefix else f"http://localhost:{RT_PORT}/"
if ready:
    print(f"\nready - interactive sim embedded below (also open in a tab: {sim_url})")
    display(IFrame(sim_url, width="100%", height=840))
else:
    print(f"\nserver not ready yet; wait a moment and re-run this cell. URL: {sim_url}")

## Key Takeaways

Now you know:
- How to load a fine-tuned MolmoAct2 policy directly and drive it in the LIBERO simulator, entirely on AMD ROCm hardware
- What a **synchronous receding-horizon** rollout is, and why it trades a brief pause between chunks for precise, blend-free motion
- How a GPU-rendered simulator is embedded inline through `jupyter-server-proxy`, with no extra port to forward

## References

* [MolmoAct2 base model (Hugging Face)](https://huggingface.co/allenai/MolmoAct2-DROID)
* [MolmoAct2 LIBERO checkpoint (Hugging Face)](https://huggingface.co/allenai/MolmoAct2-LIBERO)
* [MolmoAct2 LIBERO dataset (Hugging Face)](https://huggingface.co/datasets/allenai/MolmoAct2-LIBERO-Dataset)
* [LeRobot](https://github.com/huggingface/lerobot)

**Continue to**: [3_inference_fastwam_libero.ipynb](3_inference_fastwam_libero.ipynb)

---
Copyright© 2026 AMD, Inc SPDX-License-Identifier: MIT